In [ ]:
import sys
from unittest.mock import MagicMock
sys.modules['transformers.dependency_versions_check'] = MagicMock()

import os
import json
import torch
import torch.nn as nn
import random
import numpy as np
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoTokenizer, get_cosine_schedule_with_warmup
from peft import LoraConfig, get_peft_model, TaskType
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, matthews_corrcoef
)

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.backends.mps.is_available():
        torch.mps.manual_seed(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

seed_everything(seed=42)

In [ ]:
# ============================================================
# CHANGE THIS to run each of the 5 binary detectors:
# "Kettle", "Fridge", "WashingMachine", "Microwave", "Dishwasher"
TARGET_CLASS = "Microwave"  # Change to run for other classes
# ============================================================

MODEL_ID = "meta-llama/Llama-3.2-1B-Instruct"
FINAL_MODEL_DIR = f"./saved_binary_{TARGET_CLASS}"
CLASSES = ["Kettle", "Fridge", "WashingMachine", "Microwave", "Dishwasher"]
assert TARGET_CLASS in CLASSES

TRAIN_JSON = "temporal_train_hybrid.json"
VAL_JSON = "temporal_val_hybrid.json"
TEST_JSON = "temporal_test_hybrid.json"
FEATURE_NAMES_JSON = "hybrid_feature_names.json"

BATCH_SIZE = 8
device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Target class: {TARGET_CLASS}")
print(f"Compute target: {device.upper()}")
torch_dtype = torch.float32

In [ ]:
print("Loading dataset splits...")
with open(TRAIN_JSON, "r") as f: train_raw = json.load(f)
with open(VAL_JSON, "r") as f: val_raw = json.load(f)
with open(TEST_JSON, "r") as f: test_raw = json.load(f)
with open(FEATURE_NAMES_JSON, "r") as f: feature_names = json.load(f)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

def relabel_binary(raw_items, target_class):
    for item in raw_items:
        item["binary_label"] = 1 if item["label"] == target_class else 0
    return raw_items

train_raw = relabel_binary(train_raw, TARGET_CLASS)
val_raw = relabel_binary(val_raw, TARGET_CLASS)
test_raw = relabel_binary(test_raw, TARGET_CLASS)

train_pos = sum(item["binary_label"] for item in train_raw)
print(f"Train: {len(train_raw)} total, {train_pos} positive ({TARGET_CLASS}), {len(train_raw)-train_pos} negative (rest)")

## Standardize numeric features (fit on train only)

In [ ]:
X_train_raw = np.array([item["numeric_features"] for item in train_raw], dtype=np.float32)
X_val_raw = np.array([item["numeric_features"] for item in val_raw], dtype=np.float32)
X_test_raw = np.array([item["numeric_features"] for item in test_raw], dtype=np.float32)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_raw)
X_val_scaled = scaler.transform(X_val_raw)
X_test_scaled = scaler.transform(X_test_raw)

print("Numeric feature matrix shapes:", X_train_scaled.shape, X_val_scaled.shape, X_test_scaled.shape)

In [ ]:
class BinaryHybridDataset(Dataset):
    def __init__(self, raw_items, numeric_scaled):
        self.texts = [item["text"] for item in raw_items]
        self.binary_labels = [item["binary_label"] for item in raw_items]
        self.numeric = numeric_scaled.astype(np.float32)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return {
            "text": self.texts[idx],
            "binary_label": self.binary_labels[idx],
            "numeric": self.numeric[idx],
        }

train_dataset = BinaryHybridDataset(train_raw, X_train_scaled)
val_dataset = BinaryHybridDataset(val_raw, X_val_scaled)
test_dataset = BinaryHybridDataset(test_raw, X_test_scaled)

In [ ]:
def make_collate(include_labels=True):
    def collate_fn(batch):
        texts = [item["text"] for item in batch]
        numeric = torch.tensor(np.stack([item["numeric"] for item in batch]), dtype=torch.float32)
        encodings = tokenizer(texts, padding=True, truncation=True, max_length=512, return_tensors="pt")
        encodings["numeric"] = numeric
        if include_labels:
            encodings["labels"] = torch.tensor([item["binary_label"] for item in batch], dtype=torch.float32)
        return encodings
    return collate_fn

g = torch.Generator()
g.manual_seed(42)


val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=make_collate(True))
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=make_collate(False))
from torch.utils.data import WeightedRandomSampler

# 1. Extract all binary labels from the training raw dataset
train_labels = np.array([item["binary_label"] for item in train_raw], dtype=np.int64)

# 2. Count total occurrences of each class
class_counts = np.bincount(train_labels) # [num_negatives, num_positives]
class_weights = 1.0 / class_counts       # Invert counts to get class weights

# 3. Assign a specific weight to every single sample based on its class
sample_weights = class_weights[train_labels]
sample_weights = torch.tensor(sample_weights, dtype=torch.float32)

# 4. Create the sampler (num_samples=len(...) ensures an epoch stays the same size)
train_sampler = WeightedRandomSampler(
    weights=sample_weights, 
    num_samples=len(sample_weights), 
    replacement=True
)

# 5. Build your DataLoader (Note: shuffle MUST be False or omitted when using a sampler)
train_loader = DataLoader(
    train_dataset, 
    batch_size=BATCH_SIZE, 
    sampler=train_sampler, 
    collate_fn=make_collate(True),
    generator=g
)

# Keep your original pos_weight calculation below for loss compatibility if needed
neg_count = len(train_raw) - train_pos
pos_weight_value = neg_count / max(train_pos, 1)
pos_weight = torch.tensor([pos_weight_value], dtype=torch.float32).to(device)
print(f"BCE pos_weight: {pos_weight_value:.2f}")
neg_count = len(train_raw) - train_pos
pos_weight_value = neg_count / max(train_pos, 1)
pos_weight = torch.tensor([pos_weight_value], dtype=torch.float32).to(device)
print(f"BCE pos_weight (to counteract class imbalance): {pos_weight_value:.2f}")

In [ ]:
class FocalLossWithLogits(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.bce = nn.BCEWithLogitsLoss(reduction='none')

    def forward(self, logits, targets):
        bce_loss = self.bce(logits, targets)
        probs = torch.sigmoid(logits)
        p_t = probs * targets + (1 - probs) * (1 - targets)
        focal_weight = (1 - p_t) ** self.gamma
        loss = focal_weight * bce_loss
        
        if self.alpha >= 0:
            alpha_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
            loss = alpha_t * loss
        return loss.mean()

# Replace criterion:
# Replace your entire Focal Loss block with a clean, unweighted criterion:
criterion = nn.BCEWithLogitsLoss()

In [ ]:
class HybridBinaryClassifier(nn.Module):
    def __init__(self, llm_backbone, numeric_dim, numeric_hidden=64):
        super().__init__()
        self.llm = llm_backbone
        self.numeric_mlp = nn.Sequential(
            nn.Linear(numeric_dim, numeric_hidden),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(numeric_hidden, numeric_hidden),
            nn.ReLU(),
        )
        combined_dim = self.llm.config.hidden_size + numeric_hidden
        self.classifier = nn.Sequential(
            nn.Linear(combined_dim, combined_dim // 2),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(combined_dim // 2, 1),  # single logit for binary classification
        )

    def forward(self, input_ids, attention_mask, numeric):
        outputs = self.llm(input_ids=input_ids, attention_mask=attention_mask)
        hidden = outputs.last_hidden_state
        seq_lengths = attention_mask.sum(dim=1) - 1
        batch_idx = torch.arange(hidden.size(0), device=hidden.device)
        pooled = hidden[batch_idx, seq_lengths]

        numeric_emb = self.numeric_mlp(numeric)
        combined = torch.cat([pooled, numeric_emb], dim=-1)
        logit = self.classifier(combined).squeeze(-1)  # (batch,)
        return logit

print("Loading Llama-3.2-1B backbone...")
base_llm = AutoModel.from_pretrained(MODEL_ID, torch_dtype=torch_dtype).to(device)

lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj"],
    lora_dropout=0.2,
    bias="none",
    task_type=TaskType.FEATURE_EXTRACTION,
)
base_llm = get_peft_model(base_llm, lora_config)
base_llm.print_trainable_parameters()

model = HybridBinaryClassifier(base_llm, numeric_dim=X_train_scaled.shape[1]).to(device)
criterion = nn.BCEWithLogitsLoss()

In [ ]:
epochs = 10
# Separate pre-trained/adapted parameters from raw dense layers
llm_params = [p for n, p in model.named_parameters() if "llm" in n]
scratch_params = [p for n, p in model.named_parameters() if "numeric_mlp" in n or "classifier" in n]

optimizer = torch.optim.AdamW([
    {"params": llm_params, "lr": 5e-5, "weight_decay": 0.01},       # Gentle rate for LoRA text
    {"params": scratch_params, "lr": 1e-2, "weight_decay": 0.01}   # Higher rate for new MLP layers
])

num_training_steps = len(train_loader) * epochs
num_warmup_steps = int(0.1 * num_training_steps)
scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=num_warmup_steps, num_training_steps=num_training_steps)

best_val_loss = float("inf")
best_val_acc = 0.0  # Initialize best_val_acc before the training loop
patience = 3
patience_counter = 0

print(f"Training binary detector for: {TARGET_CLASS}")
for epoch in range(epochs):
    model.train()
    train_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        numeric = batch["numeric"].to(device)
        labels = batch["labels"].to(device)

        logits = model(input_ids=input_ids, attention_mask=attention_mask, numeric=numeric)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        scheduler.step()
        train_loss += loss.item()

    model.eval()
    val_loss = 0
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            numeric = batch["numeric"].to(device)
            labels = batch["labels"].to(device)

            logits = model(input_ids=input_ids, attention_mask=attention_mask, numeric=numeric)
            loss = criterion(logits, labels)
            val_loss += loss.item()
            preds = (torch.sigmoid(logits) > 0.5).float()
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)

    avg_train_loss = train_loss / len(train_loader)
    avg_val_loss = val_loss / len(val_loader)
    val_acc = val_correct / val_total
    current_lr = optimizer.param_groups[0]['lr']
    print(f"Epoch {epoch+1}/{epochs} | LR: {current_lr:.2e} | Train Loss: {avg_train_loss:.4f} | "
          f"Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc*100:.2f}%")
    
    

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_counter = 0
        torch.save(model.state_dict(), FINAL_MODEL_DIR + ".pt")
        print("  -> new best val accuracy, checkpoint saved.")
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print("Early stopping triggered.")
            break

print("Reloading best checkpoint...")
model.load_state_dict(torch.load(FINAL_MODEL_DIR + ".pt", map_location=device))
model.eval()

print("\nGathering predictions for threshold optimization and testing...")

# 1. Collect validation set predictions (To select the threshold)
y_val_true = []
y_val_prob = []

with torch.no_grad():
    for batch in val_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        numeric = batch["numeric"].to(device)
        labels = batch["labels"].to(device)

        logits = model(input_ids=input_ids, attention_mask=attention_mask, numeric=numeric)
        probs = torch.sigmoid(logits)
        
        y_val_true.extend(labels.cpu().numpy())
        y_val_prob.extend(probs.cpu().numpy())

# Convert validation tracking lists to native arrays
y_val_true = np.array(y_val_true)
y_val_prob = np.array(y_val_prob)


# 2. Collect test set predictions (To evaluate performance blindly)
y_true = []
y_prob = []

with torch.no_grad():
    for batch in test_loader: # Assuming you have a test_loader ready
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        numeric = batch["numeric"].to(device)
        labels = batch["labels"].to(device)

        logits = model(input_ids=input_ids, attention_mask=attention_mask, numeric=numeric)
        probs = torch.sigmoid(logits)
        
        y_true.extend(labels.cpu().numpy())
        y_prob.extend(probs.cpu().numpy())

# Convert test tracking lists to native arrays
y_true = np.array(y_true)
y_prob = np.array(y_prob)

print("✅ Successfully isolated validation and test streams.")

In [ ]:
print("\nGathering predictions for threshold optimization and testing...")

# =====================================================================
# 1. COLLECT VALIDATION DATA (To select the optimal threshold)
# =====================================================================
y_val_true = []
y_val_prob = []

# Dynamically find the label key inside the validation loader batch
first_val_batch = next(iter(val_loader))
val_label_key = "labels" if "labels" in first_val_batch else "binary_label"

with torch.no_grad():
    for batch in val_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        numeric = batch["numeric"].to(device)
        labels = batch[val_label_key].to(device)

        logits = model(input_ids=input_ids, attention_mask=attention_mask, numeric=numeric)
        probs = torch.sigmoid(logits)
        
        y_val_true.extend(labels.cpu().numpy())
        y_val_prob.extend(probs.cpu().numpy())

y_val_true = np.array(y_val_true)
y_val_prob = np.array(y_val_prob)


# =====================================================================
# 2. COLLECT TEST DATA (To evaluate performance blindly without leakage)
# =====================================================================
y_prob = []

with torch.no_grad():
    for batch in test_loader: 
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        numeric = batch["numeric"].to(device)

        logits = model(input_ids=input_ids, attention_mask=attention_mask, numeric=numeric)
        probs = torch.sigmoid(logits)
        y_prob.extend(probs.cpu().numpy())

y_prob = np.array(y_prob)

# FIX: Pull ground-truth binary targets directly from the raw test list
# This completely bypasses the KeyError since we don't look at the batch dict
y_true = np.array([item["binary_label"] for item in test_raw])

print(f"✅ Isolated Streams: {len(y_val_prob)} Val predictions vs. {len(y_prob)} Test predictions.")


## Evaluate on the held-out test set

In [ ]:
import time
y_true = [item["binary_label"] for item in test_raw]
y_prob = []
start_time=time.perf_counter()
with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        numeric = batch["numeric"].to(device)
        logits = model(input_ids=input_ids, attention_mask=attention_mask, numeric=numeric)
        probs = torch.sigmoid(logits).cpu().numpy()
        y_prob.extend(probs.tolist())
# Synchronize after finishing operations
if device == "mps":
    torch.mps.synchronize()
elif device == "cuda":
    torch.cuda.synchronize()

end_time = time.perf_counter()
total_inference_time = end_time - start_time

y_pred = [1 if p > 0.5 else 0 for p in y_prob]
matthews_corrcoef
print("=============================================")
print(f"BINARY DETECTOR RESULTS: {TARGET_CLASS} vs Rest")
print("=============================================")
print(f"Total Test Sequences : {len(y_true)}")
print(f"Accuracy              : {accuracy_score(y_true, y_pred) * 100:.2f}%")
print(f"Matthews Correlation  : {matthews_corrcoef(y_true, y_pred):.4f}")
print(f"Error rate            : {(1 - accuracy_score(y_true, y_pred)) * 100:.2f}%")
print(f"Precision             : {precision_score(y_true, y_pred, zero_division=0):.4f}")
print(f"Recall                : {recall_score(y_true, y_pred, zero_division=0):.4f}")
print(f"F1 Score              : {f1_score(y_true, y_pred, zero_division=0):.4f}")
print(f"ROC-AUC                : {roc_auc_score(y_true, y_prob):.4f}")
print(f"Total Inference Time  : {total_inference_time:.4f} seconds")
print("=============================================\\n")
print(classification_report(y_true, y_pred, target_names=[f"Not-{TARGET_CLASS}", TARGET_CLASS], zero_division=0))

cm = confusion_matrix(y_true, y_pred)
print(f"Confusion Matrix (rows=true, cols=pred), classes=[Not-{TARGET_CLASS}, {TARGET_CLASS}]:")
print(cm)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc

# 1. Compute False Positive Rate, True Positive Rate, and Thresholds
fpr, tpr, thresholds = roc_curve(y_true, y_prob)
roc_auc = auc(fpr, tpr)  # Or use your existing roc_auc_score(y_true, y_prob)

# 2. Plot the ROC Curve
plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Guess (AUC = 0.50)')

# 3. Customize Plot Labels and Limits
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (FPR)')
plt.ylabel('True Positive Rate (TPR)')
plt.title(f'ROC Curve: {TARGET_CLASS} vs Rest')
plt.legend(loc="lower right")
plt.grid(True, linestyle=':', alpha=0.6)

# 4. Display or Save the Plot
plt.tight_layout()
plt.show()
# If running headless on a server, swap plt.show() with:
# plt.savefig(f'roc_curve_{TARGET_CLASS}.png', dpi=300)

In [ ]:
# =====================================================================
# MULTIMODAL EXPLAINABILITY SYSTEM: BALANCED TEST SAMPLES BREAKDOWN
# =====================================================================
print("\n" + "="*70)
print(f"GENERATING EXPLAINABILITY BREAKDOWN FOR BALANCED TEST SEQUENCES ({TARGET_CLASS})")
print("="*70)

# 1. Dynamically locate positive and negative target instances in the test set
pos_indices = [i for i, item in enumerate(test_raw) if item["binary_label"] == 1]
neg_indices = [i for i, item in enumerate(test_raw) if item["binary_label"] == 0]

# Pick a balanced batch: up to 5 true positives and fill the rest up to 10 with negatives
num_pos_to_take = min(5, len(pos_indices))
num_neg_to_take = 10 - num_pos_to_take

sample_indices = pos_indices[:num_pos_to_take] + neg_indices[:num_neg_to_take]
model.eval()

with torch.no_grad():
    for rank_idx, idx in enumerate(sample_indices):
        # Get raw item info for text context
        raw_item = test_raw[idx]
        true_label = raw_item["binary_label"]
        appliance_name = raw_item["label"]
        prompt_text = raw_item["text"]
        
        # Parse the dynamic timeline out of the prompt text for clean printing
        timeline_snippet = "\n".join([line for line in prompt_text.split("\n") if "t=" in line])
        
        # Get individual tokenized tensors for this single sample
        single_collated = make_collate(include_labels=False)([test_dataset[idx]])
        input_ids = single_collated["input_ids"].to(device)
        attention_mask = single_collated["attention_mask"].to(device)
        numeric = single_collated["numeric"].to(device)
        
        # 2. Get the base prediction probability
        logits = model(input_ids=input_ids, attention_mask=attention_mask, numeric=numeric)
        prob = torch.sigmoid(logits).item()
        pred_label = 1 if prob > 0.5 else 0
        
        # 3. Calculate Modality Contribution (Ablation)
        # What happens if we mute the numeric features (zero them out)?
        muted_numeric = torch.zeros_like(numeric)
        text_only_logits = model(input_ids=input_ids, attention_mask=attention_mask, numeric=muted_numeric)
        text_only_prob = torch.sigmoid(text_only_logits).item()
        
        # Estimate vector contributions based on probability delta
        numeric_contribution = prob - text_only_prob
        
        # Extract dominant physical metrics directly from the raw data dictionary
        p_max_val = raw_item["numeric_features"][0]  # First feature in vector mapping
        h3_val = raw_item["numeric_features"][5]     # 3rd harmonic ratio index
        h5_val = raw_item["numeric_features"][6]     # 5th harmonic ratio index
        
        # 4. Print clean diagnostic card
        status = "✅ CORRECT" if pred_label == true_label else "❌ MISCLASSIFIED"
        print(f"\n--- DISPLAY SAMPLE #{rank_idx+1:02d} (Dataset Index {idx}) | {status} ---")
        print(f"Ground Truth : Class is '{appliance_name}' (Binary Target={true_label})")
        print(f"Model Output : Probability = {prob:.4f} -> Predicted Binary Label = {pred_label}")
        print(f"\n[Captured Timeline Signatures]:\n{timeline_snippet}")
        
        print(f"\n[Modality Influence Analysis]:")
        print(f" * Text Stream Context Baseline: {text_only_prob*100:.1f}% confidence shift.")
        if abs(numeric_contribution) > 0.05:
            direction = "INCREASED" if numeric_contribution > 0 else "DECREASED"
            print(f" * Raw Wave MLP Branch impact: {direction} target probability by {abs(numeric_contribution)*100:.1f}%.")
        else:
            print(f" * Raw Wave MLP Branch impact: Neutral / Confirmed text branch direction.")
            
        print(f"\n[Dominant Feature Decision Anchors]:")
        print(f" * Peak Onset Power (P_max) : {p_max_val:.1f} W")
        print(f" * Spectral Ratios          : 3rd Harm={h3_val:.4f} | 5th Harm={h5_val:.4f}")
        
        # Diagnostic structural logic mapping
        print("\n[AI Logic Explanation]:")
        if pred_label == 1:
            print(f" => The system flagged this as a '{TARGET_CLASS}' onset because the combination of structural text step values ")
            print(f"    and a peak load behavior profile aligned firmly with the fine-tuned LoRA parameter targets.")
        else:
            print(f" => The system rejected this as a '{TARGET_CLASS}' instance. The active spectral signature features ")
            print(f"    and structural step profiles indicate this belongs to the background ambient 'Rest' class group.")
        print("-" * 70)